> 在线讲义：[第 1 章 · 机器人策略是什么](https://wikieden.github.io/vla-from-scratch/lessons/01-what-is-a-policy.html) · 本书《VLA 从零到 π0》

# 第 1 章 · 机器人策略是什么

配套讲义：仓库里 `vla-deep-dive/notes/lessons/01-what-is-a-policy.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 动作就是一个向量 | `describe(action)` | [0.02, 0, −0.01, 0, 0, 0, 1] = Δx +2 厘米，Δz −1 厘米，夹爪合上 |
| 策略循环 | `toy_policy(...)`、`run_loop(3)` | 4 个时刻：3 次往下 1 厘米，第 4 次合上夹爪 |

动作是 OpenVLA 那种 7 维末端增量：Δx, Δy, Δz, Δroll, Δpitch, Δyaw, 夹爪（0 张开 / 1 合上）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

In [ ]:
NAMES = ["Δx", "Δy", "Δz", "Δroll", "Δpitch", "Δyaw", "夹爪"]

**函数 `describe`**：把一个 7 维动作翻译成人话，只说非零的位置。

In [ ]:
def describe(action):
    """把一个 7 维动作翻译成人话，只说非零的位置。"""
    parts = []
    for name, v in zip(NAMES, action):
        if name == "夹爪":
            parts.append("夹爪合上" if v >= 0.5 else "夹爪张开")
        elif v != 0:
            parts.append(f"{name} {v * 100:+.0f} 厘米" if name in ("Δx", "Δy", "Δz") else f"{name} {v:+.2f}")
    return "，".join(parts)

**函数 `toy_policy`**：玩具策略 π(o, ℓ)：指令"抓起杯子"。离杯子还有高度就往下 1 厘米，到了就合上夹爪。

In [ ]:
def toy_policy(height_cm, gripper_closed):
    """玩具策略 π(o, ℓ)：指令"抓起杯子"。离杯子还有高度就往下 1 厘米，到了就合上夹爪。"""
    if height_cm > 0:
        return np.array([0, 0, -0.01, 0, 0, 0, float(gripper_closed)])
    return np.array([0, 0, 0, 0, 0, 0, 1.0])

**函数 `run_loop`**：策略循环：看 → 想 → 做，做完世界就变了，再看。

In [ ]:
def run_loop(height_cm):
    """策略循环：看 → 想 → 做，做完世界就变了，再看。"""
    actions, closed = [], False
    while not closed:
        a = toy_policy(height_cm, closed)
        actions.append(a)
        height_cm = round(height_cm + a[2] * 100)  # 做：手往下走，下一眼看到的高度变了
        closed = a[6] >= 0.5
    return actions

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 动作就是一个向量")
a = np.array([0.02, 0, -0.01, 0, 0, 0, 1])
print(f"  {a.tolist()} = {describe(a)}")

In [ ]:
print("\n第二步 策略循环：一个时刻只输出一个动作")
for t, act in enumerate(run_loop(3), 1):
    print(f"  时刻 {t}：{act.tolist()}（{describe(act)}）")

## 3. 代码里才看得到的

1. `run_loop` 每走一步都重新"看"一次高度：做完世界就变了，下一步的输入不同。
2. 夹爪用 0 / 1 表示，大于等于 0.5 当成合上；真实数据里夹爪也常是连续值。

## 4. 自己改一改

- 把 `toy_policy` 改成一次往下 2 厘米，数一数要几个时刻。
- 给动作加一个 Δx，看 `describe` 怎么翻译。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 看一眼，听一句，吐一串数；一秒吐很多次，吐完世界就变了。